# R2AI 2026 — Inference 2000 câu trên Kaggle GPU (có AUTO-TUNE)

Pipeline thật (BM25 + Dense → RRF → Reranker → F2-select), chạy GPU. **Một lần Run All** sẽ:
dò file → sinh config → **tự tune `max_k`/`rel_threshold` trên dev set** → chạy full 2000 → đóng gói `submission.zip`.

**Vì sao thêm auto-tune:** điểm thấp lần đầu (Articles F2 ≈ 0.41) gốc rễ là `rel_threshold=0.3` quá lỏng → precision tụt. Cell 3b đo F2 thật trên dev rồi ghi đè 2 tham số selection trước khi chạy full, nên không phải đoán tay.

**Trước khi Run All:**
1. Panel phải → **Accelerator = GPU T4 ×2** (hoặc P100), **Internet = On** (tải model HF).
2. **Add Data** → dataset phải có:
   - index: `bm25.pkl`, `dense.pkl`
   - code repo: `src/`, `scripts/` (gồm **2 script mới**: `dump_candidates.py`, `tune_from_cache.py`), `config*.yaml`
   - test: `R2AIStage1DATA.json`
   - **dev set (cho auto-tune):** `dev_test.json` + `dev_gold.json` *(thiếu → notebook bỏ qua tune, dùng mặc định 8/0.3)*
   - Notebook tự dò đường dẫn bằng glob nên không cần sửa slug.
3. **Run All**. `results.json` + `submission.zip` ở tab **Output**.

**Thứ tự cell:** 1 env · 2 dò file · 3 config · **3b auto-tune (mới)** · 4 smoke *(tuỳ chọn)* · 5 full 2000 · 6 validate+zip.

In [ ]:
# Cell 1 — GPU + deps + tắt log spam + helper chạy subprocess KHÔNG bị nhân đôi
import os, subprocess, sys

# Tắt progress bar / log thừa của HuggingFace + transformers
# (đây là thứ sinh ra mấy nghìn dòng "Loading weights ... Materializing param")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# HF_TOKEN từ Kaggle Secrets (nếu đã add) -> tránh rate-limit khi tải model
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN: chua set (van chay duoc, chi rate-limit thap hon)")

def run(cmd):
    """Chay subprocess va stream output DUNG 1 LAN qua sys.stdout cua notebook.
    Gop stderr -> stdout de tqdm khong tach dong. Khac phuc viec Kaggle in
    nhan doi moi dong tu child process (do child ke thua raw fd 1)."""
    # -u: ep child python in KHONG buffer -> log N/2000 hien LIVE thay vi
    # ket trong block-buffer cho toi khi process xong.
    if cmd and cmd[0] == sys.executable and '-u' not in cmd:
        cmd = [cmd[0], '-u'] + list(cmd[1:])
    env = dict(os.environ); env['PYTHONUNBUFFERED'] = '1'
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    return p.returncode

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                    capture_output=True, text=True).stdout or "KHONG thay GPU — bat Accelerator!")
run([sys.executable, "-m", "pip", "install", "-q", "pyvi", "rank-bm25", "rapidfuzz"])
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())


In [ ]:
# Cell 2 — tự dò code repo, index, test file dưới /kaggle/input; copy code sang /kaggle/working (ghi được)
import glob, os, shutil

def find_one(pattern, what):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f"KHÔNG tìm thấy {what} (pattern: {pattern}). Kiểm tra Add Data."
    return hits[0]

def find_opt(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

bm25_pkl  = find_one("/kaggle/input/**/bm25.pkl", "bm25.pkl")
dense_pkl = find_one("/kaggle/input/**/dense.pkl", "dense.pkl")
cfg_py    = find_one("/kaggle/input/**/src/config.py", "src/config.py (code repo)")
test_json = find_one("/kaggle/input/**/R2AIStage1DATA.json", "R2AIStage1DATA.json")

# dev set cho auto-tune (KHÔNG bắt buộc — nếu thiếu thì bỏ qua bước tune)
dev_test_json = find_opt("/kaggle/input/**/dev_test.json")
dev_gold_json = find_opt("/kaggle/input/**/dev_gold.json")

index_dir_src = os.path.dirname(bm25_pkl)
code_root_src = os.path.dirname(os.path.dirname(cfg_py))   # thư mục chứa src/ & scripts/
print("index_dir :", index_dir_src)
print("code_root :", code_root_src)
print("test_file :", test_json)
print("dev_test  :", dev_test_json)
print("dev_gold  :", dev_gold_json)

# Copy code (nhẹ) sang working để ghi config + chạy; index giữ nguyên ở input (read-only OK khi load)
REPO = "/kaggle/working/r2ai_pipeline"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(code_root_src, REPO, ignore=shutil.ignore_patterns(
    ".venv", "__pycache__", "*.pkl", "data"))
print("copied code ->", REPO)

In [ ]:
# Cell 3 — sinh config.kaggle.yaml: device=cuda, fp16=true, đường dẫn TUYỆT ĐỐI trỏ về index trong /kaggle/input
# >>> CHIẾN LƯỢC 1420 (LỎNG) — config THẮNG public (Articles F2=0.4239). KHÔNG tune theo dev:
#     dev KHÔNG đại diện public (dev=0.5731 nhưng public=0.42); tune dev (run 1545) làm TỤT public -> 0.4178.
#     => giữ max_k=8, rel_threshold=0.3, transform=none và TẮT Cell 3b (DISABLE_TUNE=True).
cfg_yaml = f"""# R2AI 2026 — Kaggle GPU inference (auto-generated)
paths:
  corpus_dir: data/corpus.json
  test_file: {test_json}
  index_dir: {index_dir_src}        # tuyệt đối -> load bm25.pkl + dense.pkl từ input
  results_file: /kaggle/working/results.json
  submission_zip: /kaggle/working/submission.zip
chunk:
  split_long: true
  max_chars: 1200
retrieval:
  bm25_top_n: 100
  dense_top_n: 100
  rrf_weights: [0.3, 0.7]
  rrf_k: 30
  rerank_top_n: 60
  use_reranker: true
  use_hyde: false
  min_k: 1
  max_k: 8              # <- 1420 LỎNG (public-best). KHÔNG để Cell 3b ghi đè.
  rel_threshold: 0.3    # <- 1420 LỎNG: giữ nhiều Điều -> ưu tiên recall (F2 trọng recall)
  rel_score_transform: none
models:
  dense_backend: st
  dense_model: AITeamVN/Vietnamese_Embedding
  dense_max_seq_length: 512
  dense_batch_size: 64
  dense_fp16: true            # T4 = fp16 OK -> encode query nhanh hơn
  reranker_backend: cross-encoder
  reranker_model: AITeamVN/Vietnamese_Reranker
  llm_backend: extractive     # lần nộp đầu: không cần LLM, answer tự trích Điều X
  llm_model: qwen3:8b
  device: cuda                # <-- mấu chốt: GPU thay vì mps
"""
cfg_path = f"{REPO}/config.kaggle.yaml"
with open(cfg_path, "w") as f:
    f.write(cfg_yaml)
print(cfg_yaml)

In [ ]:
# Cell 3b — AUTO-TUNE selection params (ĐANG TẮT theo chiến lược 1420)
# >>> DISABLE_TUNE=True: BỎ QUA tune theo dev, GIỮ NGUYÊN config 1420 lỏng ở Cell 3.
#     Lý do: dev KHÔNG đại diện public. Lần 1545 tune theo dev -> public TỤT 0.4239 -> 0.4178.
#     Với index MỚI (data đã vá: Lao động VI + Đấu thầu đủ), đòn nâng điểm là RECALL từ data,
#     không phải siết selection. Cấu hình lỏng (max_k=8, thr=0.3) ưu tiên recall = cược an toàn.
#     Muốn bật lại tune (KHÔNG khuyến nghị): đặt DISABLE_TUNE = False.
import os, sys, re
os.chdir(REPO)

DISABLE_TUNE = True   # <<< TẮT auto-tune. Giữ params 1420 từ Cell 3.

if DISABLE_TUNE:
    cur = open(cfg_path).read()
    mk  = re.search(r"max_k:\s*(\d+)", cur)
    thr = re.search(r"rel_threshold:\s*([\d.]+)", cur)
    tr  = re.search(r"rel_score_transform:\s*(\w+)", cur)
    print(">>> AUTO-TUNE TẮT (DISABLE_TUNE=True) — dùng nguyên config 1420 lỏng từ Cell 3:")
    print(f"    max_k={mk.group(1) if mk else '?'} | "
          f"rel_threshold={thr.group(1) if thr else '?'} | "
          f"rel_score_transform={tr.group(1) if tr else '?'}")
    print(">>> LƯU Ý: Cell 3b không còn smoke-test. Muốn thử nhanh trước full -> chạy Cell 4 (5 câu).")
else:
    # ===== (giữ lại) lối tune theo dev — chỉ chạy khi DISABLE_TUNE=False =====
    if not (dev_test_json and dev_gold_json):
        raise RuntimeError("Bật tune nhưng thiếu dev_test.json/dev_gold.json trong /kaggle/input.")
    rc = run([sys.executable, "scripts/dump_candidates.py",
              "--backend", "reranker", "--config", "config.kaggle.yaml",
              "--dev-test", dev_test_json, "--top-n", "60",
              "--out", "/kaggle/working/dev_cands.json"])
    assert rc == 0, "dump_candidates.py FAILED — kiểm tra dataset có 2 script mới chưa."
    run([sys.executable, "scripts/tune_from_cache.py",
         "--cand", "/kaggle/working/dev_cands.json",
         "--gold", dev_gold_json, "--write-config", "/kaggle/working/tuned.yaml"])
    tuned = open("/kaggle/working/tuned.yaml").read()
    mk  = re.search(r"max_k:\s*(\d+)", tuned)
    thr = re.search(r"rel_threshold:\s*([\d.]+)", tuned)
    tr  = re.search(r"rel_score_transform:\s*(\w+)", tuned)
    transform_val = tr.group(1) if tr else ("sigmoid" if "transform=sigmoid" in tuned else "none")
    if not (mk and thr):
        raise RuntimeError("Không parse được tuned.yaml — DỪNG để khỏi nộp params lỏng.")
    cfg_txt = open(cfg_path).read()
    cfg_txt = re.sub(r"max_k:\s*\d+.*",            f"max_k: {mk.group(1)}",          cfg_txt)
    cfg_txt = re.sub(r"rel_threshold:\s*[\d.]+.*", f"rel_threshold: {thr.group(1)}", cfg_txt)
    if re.search(r"rel_score_transform:", cfg_txt):
        cfg_txt = re.sub(r"rel_score_transform:\s*\w+.*",
                         f"rel_score_transform: {transform_val}", cfg_txt)
    else:
        cfg_txt = re.sub(r"(rel_threshold:\s*[\d.]+.*)",
                         r"\1\n  rel_score_transform: " + transform_val, cfg_txt)
    open(cfg_path, "w").write(cfg_txt)
    print(f"\n>>> ĐÃ ÁP TUNED: max_k={mk.group(1)}, rel_threshold={thr.group(1)}, "
          f"rel_score_transform={transform_val}")

In [ ]:
# Cell 4 (TUỲ CHỌN — BỎ QUA nếu Cell 3b auto-tune đã chạy xong, vì nó đã load model + verify pipeline)
# Chỉ dùng khi bạn SKIP auto-tune (không có dev set) và muốn smoke-test 5 câu trước khi chạy full.
import os, sys
os.chdir(REPO)
rc = run([sys.executable, "scripts/run_inference.py",
          "--config", "config.kaggle.yaml",
          "--test", test_json,
          "--out", "/kaggle/working/test5.json", "--limit", "5"])
print("return code:", rc)

In [ ]:
# Cell 5 — FULL inference 2000 cau, SONG SONG tren 2x T4 (~8-15')
# Chia 2000 cau cho 2 worker: GPU0 lam shard 1/2, GPU1 lam shard 2/2 -> merge.
# Neu chi co 1 GPU -> tu dong chay 1 worker (shard 1/1).
import os, sys, time, json, threading, subprocess, glob

os.chdir(REPO)
import torch
n_gpu = torch.cuda.device_count()
print(f"[infer] thay {n_gpu} GPU")

def _worker(gpu_id, shard, out_path, sink):
    env = dict(os.environ)
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)   # pin worker vao dung 1 T4
    env["PYTHONUNBUFFERED"] = "1"
    cmd = [sys.executable, "-u", "scripts/run_inference.py",
           "--config", "config.kaggle.yaml", "--test", test_json,
           "--shard", shard, "--out", out_path]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=env)
    for line in p.stdout:
        sink.append(line)
        print(f"[GPU{gpu_id}] {line}", end="")
    p.wait()
    sink.append(f"__RC__{p.returncode}")

t0 = time.time()
parts = []
if n_gpu >= 2:
    specs = [(0, "1/2", "/kaggle/working/results.part1.json"),
             (1, "2/2", "/kaggle/working/results.part2.json")]
else:
    specs = [(0, "1/1", "/kaggle/working/results.part1.json")]

sinks = [[] for _ in specs]
threads = []
for (gpu, shard, out), sink in zip(specs, sinks):
    th = threading.Thread(target=_worker, args=(gpu, shard, out, sink))
    th.start(); threads.append(th)
for th in threads:
    th.join()

# kiem tra returncode tat ca worker
for (gpu, shard, out), sink in zip(specs, sinks):
    rc = next((int(x[5:]) for x in sink if x.startswith("__RC__")), -1)
    if rc != 0:
        raise RuntimeError(f"Worker GPU{gpu} shard {shard} FAIL rc={rc} — xem log tren.")
    parts.append(out)

# --- MERGE cac shard -> results.json (sort theo id) ---
merged = {}
for pth in parts:
    for r in json.load(open(pth)):
        merged[r["id"]] = r
results = [merged[k] for k in sorted(merged)]
json.dump(results, open("/kaggle/working/results.json", "w"),
          ensure_ascii=False, indent=2)
print(f"\n[infer] MERGE {len(results)} cau tu {len(parts)} shard "
      f"-> /kaggle/working/results.json | tong {round(time.time()-t0,1)}s")


In [ ]:
# Cell 6 — validate schema + dong goi submission.zip (zip phang) + xem thu 1 record
import os, sys, json
os.chdir(REPO)
print("=== validate ===")
run([sys.executable, "scripts/validate_submission.py",
     "--results", "/kaggle/working/results.json", "--test", test_json])
print("\n=== package ===")
run([sys.executable, "scripts/make_submission.py",
     "--results", "/kaggle/working/results.json", "--out", "/kaggle/working/submission.zip"])
print("\n=== sample record ===")
d = json.load(open("/kaggle/working/results.json"))
print("total:", len(d))
print(json.dumps(d[0], ensure_ascii=False, indent=2)[:1500])


## Sau khi chạy
Tab **Output** → tải `submission.zip` về máy → upload lên **leaderboard.aiguru.com.vn**.

Lưu ý công bằng: nếu `submission.zip` của `make_submission.py` bị lồng thư mục, giải nén lại `results.json` rồi zip phẳng thủ công: `cd /kaggle/working && zip -j submission.zip results.json`.